# 🚀 Fase 6: Sktime vs The World (O Duelo Final)

A pedido do usuário, vamos reviver o laboratório de Engenharia de Features para incluir a biblioteca oficial `sktime`.
O objetivo aqui é responder à seguinte pergunta: **A abstração elegante e automatizada do `WindowSummarizer` do `sktime` consegue extrair features estatísticas o suficiente para bater o nosso modelo Híbrido Artesanal construído com Wavelets?**

## 1. Carregamento e Preparação dos Dados


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Importando a jóia da coroa do sktime para Feature Engineering Automática
from sktime.transformations.series.summarize import WindowSummarizer

plt.style.use('dark_background')

# 1. Carregar Dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00381/PRSA_data_2010.1.1-2014.12.31.csv"
df_raw = pd.read_csv(url)

# 2. Preprocessamento Diário
df_raw['date'] = pd.to_datetime(df_raw[['year', 'month', 'day', 'hour']])
cols_of_interest = ['pm2.5', 'DEWP', 'TEMP', 'PRES', 'Iws']
df_raw = df_raw[['date'] + cols_of_interest].copy()
df_raw['pm2.5'] = df_raw['pm2.5'].interpolate(method='linear')
df_raw = df_raw.dropna()

df_daily = df_raw.set_index('date').resample('D').mean().reset_index()
df_daily['target'] = df_daily['pm2.5'].shift(-1)
df_daily = df_daily.dropna().set_index('date')



## 2. Automação Elegante com `WindowSummarizer` (`sktime`)

Em vez de escrever loops e usar `.shift()` ou `.rolling()` manualmente no Pandas, o `sktime` permite mapear uma vasta gama de operações estatísticas sobre janelas temporais de forma declarativa.


In [2]:
print("Iniciando extração de features via Sktime...")
start_time = time.time()

# Configurando o gerador de features (Lags e Janelas Móveis)
# Queremos extrair: Lags diretos (1,2,3,7), e para a janela de 7 dias extrair Média, Desvio Padrão, Mínimo e Máximo
kwargs = {
    "lag_feature": {
        "lag": [1, 2, 3, 7],
        "mean": [[1, 7], [1, 14]],
        "std": [[1, 7], [1, 14]],
        "min": [[1, 7]],
        "max": [[1, 7]],
    }
}

transformer = WindowSummarizer(**kwargs, n_jobs=-1)

# Sktime processa automaticamente todas as colunas de interesse
X_sktime = transformer.fit_transform(df_daily[cols_of_interest])

# Dropando as colunas originais atuais (já que queremos focar nas janelas móveis geradas e evitar vazamento)
X_sktime = X_sktime.dropna()
y = df_daily.loc[X_sktime.index, 'target']

time_sktime = time.time() - start_time
print(f"Extração concluída em {time_sktime:.2f}s!")
print(f"O sktime gerou magicamente {X_sktime.shape[1]} features temporais.")



Iniciando extração de features via Sktime...


Extração concluída em 1.08s!
O sktime gerou magicamente 14 features temporais.


## 3. O Treinamento (Random Forest)

Vamos manter as regras justas: usar o mesmo Random Forest `default` e o mesmo conjunto de testes de 365 dias.


In [3]:
test_size = 365

X_train = X_sktime.iloc[:-test_size]
y_train = y.iloc[:-test_size]
X_test = X_sktime.iloc[-test_size:]
y_test = y.iloc[-test_size:]

print(f"Treinando Random Forest nas {X_train.shape[1]} features geradas pelo sktime...")
start_time = time.time()
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
train_time = time.time() - start_time

preds = rf.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\n--- RESULTADOS SKTIME ---")
print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"Tempo de Treino: {train_time:.2f}s")



Treinando Random Forest nas 14 features geradas pelo sktime...



--- RESULTADOS SKTIME ---
MAE: 52.7957
RMSE: 70.9945
Tempo de Treino: 0.32s


## 4. Conclusão da Fase 6

Neste experimento conseguimos um **MAE de {coloque_o_resultado_aqui}**.
Isso significa que o `sktime` é uma biblioteca EXCELENTE em termos de sintaxe e limpeza de código. Contudo, em termos de precisão pura:
- **Híbrido (Wavelets + Manual FE) - Fase 4:** MAE 54.19
- **Sktime Automated FE - Fase 6:** [A aguardar execução]

Se o MAE do Sktime ficar acima de 54.19, prova-se novamente que abstrações estatísticas clássicas (lags/rolling) não superam o processamento físico de sinais (Wavelets) para fenômenos metereológicos complexos!
